In [2]:
import netCDF4

file_path = "S5P_NRTI_L2__CLOUD__20251123T192435_20251123T192935_42043_03_020701_20251123T195759.nc"
nc = netCDF4.Dataset(file_path, mode='r')

    
for group in nc.groups:
    print(group)
    print(nc.groups[group].variables)

PRODUCT
{'scanline': <class 'netCDF4.Variable'>
int32 scanline(scanline)
    _FillValue: -2147483647
    units: 1
    axis: Y
    long_name: along-track dimension index
    comment: This coordinate variable defines the indices along track; index starts at 0
path = /PRODUCT
unlimited dimensions: 
current shape = (357,)
filling on, 'ground_pixel': <class 'netCDF4.Variable'>
int32 ground_pixel(ground_pixel)
    _FillValue: -2147483647
    units: 1
    axis: X
    long_name: across-track dimension index
    comment: This coordinate variable defines the indices across track, from west to east; index starts at 0
path = /PRODUCT
unlimited dimensions: 
current shape = (450,)
filling on, 'time': <class 'netCDF4.Variable'>
int32 time(time)
    _FillValue: -2147483647
    units: seconds since 2010-01-01 00:00:00
    standard_name: time
    axis: T
    long_name: reference time for the measurements
    comment: The time in this variable corresponds to the time in the time_reference global attribut

In [18]:
from datetime import datetime
import netCDF4
import numpy as np

# Query point
lat_q = 30.048103859530674
lon_q = -90.03008110359737

def haversine(lat1, lon1, lat2, lon2):
    # Toate în grade decimale
    R = 6371  # raza Pământului în km
    lat1 = np.deg2rad(lat1)
    lat2 = np.deg2rad(lat2)
    dlat = lat2 - lat1
    dlon = np.deg2rad(lon2 - lon1)
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    return R * 2 * np.arcsin(np.sqrt(a))

file_path = "S5P_NRTI_L2__CLOUD__20251123T192435_20251123T192935_42043_03_020701_20251123T195759.nc"
nc = netCDF4.Dataset(file_path, mode='r')

# PRODUCT group
grp = nc.groups['PRODUCT']

# Read latitude/longitude arrays: shape (time, scanline, ground_pixel)
lat = grp.variables['latitude'][:]
lon = grp.variables['longitude'][:]

# Distance (Euclidean in degrees)
dist = np.sqrt((lat - lat_q)**2 + (lon - lon_q)**2)

# Base nearest index across full swath
base_idx = np.unravel_index(np.argmin(dist), dist.shape)

# Variables to extract
vars_to_extract = [
    "cloud_fraction",
    "cloud_top_pressure",
    "cloud_base_pressure",
    "cloud_top_height",
    "cloud_base_height",
    "cloud_optical_thickness",
    "delta_time"
]

def is_masked(arr, idx):
    """Return True if arr[idx] is masked. Works for both masked and plain ndarrays."""
    v = arr[idx]
    if np.ma.isMaskedArray(arr):
        return np.ma.getmaskarray(arr)[idx]
    # Some files encode invalids as NaN; treat NaN as masked-like.
    return np.isnan(v)

def find_nearest_unmasked_for_var(var_arr, lat, lon, qlat, qlon, base_idx, max_distance_km=10, max_window=50):
    """
    Caută cel mai apropiat pixel NEMASCAT pentru această variabilă, DAR numai dacă se află la <= max_distance_km!
    """
    R = 6371
    def fast_haversine(latp, lonp, lat0, lon0):
        # vectorizat pentru arrays mici 2D
        latp = np.deg2rad(latp)
        lat0 = np.deg2rad(lat0)
        dlat = latp - lat0
        dlon = np.deg2rad(lonp - lon0)
        a = np.sin(dlat/2.0)**2 + np.cos(latp)*np.cos(lat0)*np.sin(dlon/2.0)**2
        return R * 2 * np.arcsin(np.sqrt(a))

    # Ca și înainte, doar că distanța este km și filtrăm la max_distance_km
    if not is_masked(var_arr, base_idx):
        # verif distanța
        t0 = base_idx[0]
        d = fast_haversine(lat[base_idx], lon[base_idx], qlat, qlon)
        if d <= max_distance_km:
            return base_idx, var_arr[base_idx]
        else:
            return None, None

    tdim, sdim, gdim = var_arr.shape
    t0 = base_idx[0]
    lat_t = lat[t0]
    lon_t = lon[t0]
    dist_t = fast_haversine(lat_t, lon_t, qlat, qlon)

    s0, g0 = base_idx[1], base_idx[2]

    best_idx = None
    best_dist = np.inf

    if np.ma.isMaskedArray(var_arr):
        mask_t = np.ma.getmaskarray(var_arr[t0])
    else:
        mask_t = np.isnan(var_arr[t0])

    for r in range(0, max_window + 1):
        s_min = max(0, s0 - r)
        s_max = min(sdim - 1, s0 + r)
        g_min = max(0, g0 - r)
        g_max = min(gdim - 1, g0 + r)

        sub_mask = mask_t[s_min:s_max+1, g_min:g_max+1]
        sub_dist = dist_t[s_min:s_max+1, g_min:g_max+1]

        # Considerăm doar unde valid și sub distanța maximă
        valid = (~sub_mask) & (sub_dist <= max_distance_km)
        if np.any(valid):
            local_argmin = np.argmin(np.where(valid, sub_dist, np.inf))
            local_idx2d = np.unravel_index(local_argmin, sub_dist.shape)
            s_idx = s_min + local_idx2d[0]
            g_idx = g_min + local_idx2d[1]
            d = sub_dist[local_idx2d]
            if d < best_dist:
                best_dist = d
                best_idx = (t0, s_idx, g_idx)

    # CAUTARE fallback în toată imaginea DOAR dacă vrei (atenție, poate fi slow)
    if best_idx is None:
        for t in range(tdim):
            if np.ma.isMaskedArray(var_arr):
                mask_t = np.ma.getmaskarray(var_arr[t])
            else:
                mask_t = np.isnan(var_arr[t])
            dist_t = fast_haversine(lat[t], lon[t], qlat, qlon)
            valid = (~mask_t) & (dist_t <= max_distance_km)
            if np.any(valid):
                local_argmin = np.argmin(np.where(valid, dist_t, np.inf))
                idx3d = (t,) + np.unravel_index(local_argmin, dist_t.shape)
                d = dist_t.flat[local_argmin]
                if d < best_dist:
                    best_dist = d
                    best_idx = idx3d

    if best_idx is None:
        return None, None
    else:
        return best_idx, var_arr[best_idx]

# Extract values for each variable from the nearest unmasked pixel for that variable
results = {}

for var in vars_to_extract:
    arr = grp.variables[var][:]
    best_idx, val = find_nearest_unmasked_for_var(arr, lat, lon, lat_q, lon_q, base_idx, max_distance_km=10, max_window=50)
    
    if "Time" not in results:
        utc = grp.variables['time_utc'][:]   # shape (time, scanline)
        # Suppose best_idx = (time_idx, scanline_idx, ground_pixel_idx)
        current_utc = utc[best_idx[0], best_idx[1]]  # String: e.g. '2025-11-23T19:24:35.123Z'
        if '.' in current_utc:
            current_utc = current_utc.split('.')[0]
        current_utc = current_utc.replace('Z','')
        
        # Parsem stringul și formatez
        dt = datetime.strptime(current_utc, "%Y-%m-%dT%H:%M:%S")
        formatted = dt.strftime("%m/%d/%Y %#H:%M")                
        results["Time"] = {"value": formatted}
    
    if best_idx is None:
        results[var] = {"index": None, "value": None, "note": "No unmasked value found within 10 km"}
    else:
        results[var] = {"index": best_idx, "value": float(val) if np.isscalar(val) else val}
        

# Print results
for var, info in results.items():
    print(f"{var}: value={info['value']}")

Time: value=11/23/2025 19:29
cloud_fraction: value=0.0
cloud_top_pressure: value=73781.3671875
cloud_base_pressure: value=82118.5546875
cloud_top_height: value=2845.841064453125
cloud_base_height: value=1845.8411865234375
cloud_optical_thickness: value=7.258948802947998
delta_time: value=70141645.0


In [34]:
import netCDF4

f = netCDF4.Dataset("SST/23e19fd7dd60fb7555b2c634054f5bd4.nc")
print(f.variables.keys())  # Listează toate variabilele


dict_keys(['number', 'valid_time', 'latitude', 'longitude', 'expver', 'sst'])


In [37]:
import netCDF4
import numpy as np
from datetime import datetime, timedelta
import pandas as pd

lat_q = 30.048103859530674
lon_q = -90.03008110359737

ds = netCDF4.Dataset("SST/7a0b702d34c382842339087fe44ac453.nc")
lats = ds.variables["latitude"][:]
lons = ds.variables["longitude"][:]
tcwv = ds.variables["sst"][:]         # shape: (time, lat, lon)
times = ds.variables["valid_time"][:]

lat_idx = np.abs(lats - lat_q).argmin()
lon_idx = np.abs(lons - lon_q).argmin()

# Pregătește liste pentru DataFrame
time_list = []
tcwv_list = []

epoch = datetime(1970, 1, 1)

# for i, t in enumerate(times):
#     t_dt = epoch + timedelta(seconds=float(t))
#     t_str = t_dt.strftime('%m/%d/%Y %H:%M')
#     tcwv_val = float(tcwv[i, lat_idx, lon_idx])
#     time_list.append(t_str)
#     tcwv_list.append(tcwv_val)

for i, t in enumerate(times):
    t_dt = epoch + timedelta(seconds=float(t))
    t_str = t_dt.strftime('%m/%d/%Y %H:%M')
    sst_val = tcwv[i, lat_idx, lon_idx]
    # Check if the value is masked (for MaskedArray)
    if np.ma.is_masked(sst_val):
        sst_val = np.nan
    else:
        sst_val = float(sst_val)
    time_list.append(t_str)
    tcwv_list.append(sst_val)

df = pd.DataFrame({'Time': time_list, 'SST': tcwv_list})

# print(df.head())
df.to_csv("SST/sst-2018-2019.csv", index=False)

https://cds.climate.copernicus.eu/cdsapp#!/dataset/reanalysis-era5-single-levels

In [27]:
import cdsapi

dataset = "reanalysis-era5-single-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": [
        "total_column_water"
    ],
    "year": [ "2018", "2019",
        "2020", "2021", "2022",
        "2023", "2024"
    ],
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "time": [
        "00:00", "01:00", "02:00",
        "03:00", "04:00", "05:00",
        "06:00", "07:00", "08:00",
        "09:00", "10:00", "11:00",
        "12:00", "13:00", "14:00",
        "15:00", "16:00", "17:00",
        "18:00", "19:00", "20:00",
        "21:00", "22:00", "23:00"
    ],
    "data_format": "grib",
    "download_format": "unarchived",
    "area": [31, -92, 29, -88]
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()

2025-11-29 00:12:17,028 INFO Request ID is 4c81f70c-0654-416e-997a-184021a6ffa1
2025-11-29 00:12:17,111 INFO status has been updated to accepted


KeyboardInterrupt: 

In [3]:
import xarray as xr

# Deschide fișierul GRIB
ds = xr.open_dataset("807e77de5250dfa67a0521497308cb64.grib", engine="cfgrib")

# Listează toate variabilele din fișier
print("Variabile disponibile în fișier:")
for var in ds.data_vars:
    print(var)

ValueError: unrecognized engine 'cfgrib' must be one of your download engines: ['netcdf4', 'scipy', 'store']. To install additional dependencies, see:
https://docs.xarray.dev/en/stable/user-guide/io.html 
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html